# Evaluate Trained CNNs

In [24]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

def build_model(krnl, drop):
    model = nn.Sequential(
    nn.Conv1d(32, 64, kernel_size=krnl, padding=krnl//2),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.MaxPool1d(2),

    nn.Conv1d(64, 128, kernel_size=krnl, padding=krnl//2),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.MaxPool1d(2),

    nn.Conv1d(128, 256, kernel_size=krnl, padding=krnl//2),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.AdaptiveAvgPool1d(1),
    
    nn.Flatten(),
    nn.Linear(256, 64),
    nn.ReLU(),
    nn.Dropout(drop),
    nn.Linear(64, 2))
    return model

model30 = build_model(7, .25)
model30.load_state_dict(torch.load('30_sample_model.pth'))

model20 = build_model(7, .25)
model20.load_state_dict(torch.load('20_sample_model.pth'))

model10 = build_model(7, .25)
model10.load_state_dict(torch.load('10_sample_model.pth'))

data = np.load('MEEG_windowz.npz')
print(list(data.keys()))

['x30_train', 'y30_train', 'x30_test', 'y30_test', 'x30_inSample', 'y30_inSample', 'x20_train', 'y20_train', 'x20_test', 'y20_test', 'x20_inSample', 'y20_inSample', 'x10_train', 'y10_train', 'x10_test', 'y10_test', 'x10_inSample', 'y10_inSample']


In [46]:
def eval_model(model, xData, yData):
    X = torch.tensor(xData, dtype=torch.float32)
    Y = torch.tensor(yData, dtype=torch.float32)
    test = TensorDataset(X, Y)
    loader = DataLoader(test,batch_size=256)
    
    valence_acc = 0
    arousal_acc = 0 
    xact_acc = 0
    total = 0
    model.eval()
    
    with torch.no_grad():
        for x, y in loader:
            preds = (torch.sigmoid(model(x)) >= 0.5)
            valence_acc += (preds[:, 0] == y[:, 0]).sum().item()
            arousal_acc += (preds[:, 1] == y[:, 1]).sum().item()
            xact_acc += (preds == y).all(dim=1).sum().item()
            total += x.size(0)
    
    valence_acc = valence_acc / total
    arousal_acc = arousal_acc / total
    xact_acc = xact_acc / total
    print(f'Valence Accuracy = {valence_acc:.3f}')
    print(f'Arousal Accuracy = {arousal_acc:.3f}')
    print(f'Exact Accuracy = {xact_acc:.3f}')

In [52]:
print('30 Subject Model')
print('------------------------------------------')
print('Same Subjects Test:')
eval_model(model30,data['x30_inSample'],data['y30_inSample'])
print('----------------')
print('New Subjects Test:')
eval_model(model30,data['x30_test'],data['y30_test'])

print('\n20 Subject Model')
print('------------------------------------------')
print('Same Subjects Test:')
eval_model(model20,data['x20_inSample'],data['y20_inSample'])
print('----------------')
print('New Subjects Test:')
eval_model(model20,data['x20_test'],data['y20_test'])

print('\n10 Subject Model')
print('------------------------------------------')
print('Same Subjects Test:')
eval_model(model10,data['x10_inSample'],data['y10_inSample'])
print('----------------')
print('New Subjects Test:')
eval_model(model10,data['x10_test'],data['y10_test'])


30 Subject Model
------------------------------------------
Same Subjects Test
Valence Accuracy = 0.250
Arousal Accuracy = 0.928
Exact  Accuracy = 0.235
----------------
New Subjects Test
Valence Accuracy = 0.586
Arousal Accuracy = 0.965
Exact  Accuracy = 0.576

20 Subject Model
------------------------------------------
Same Subjects Test
Valence Accuracy = 0.966
Arousal Accuracy = 0.627
Exact  Accuracy = 0.608
----------------
New Subjects Test
Valence Accuracy = 0.595
Arousal Accuracy = 0.678
Exact  Accuracy = 0.405

10 Subject Model
------------------------------------------
Same Subjects Test
Valence Accuracy = 0.527
Arousal Accuracy = 0.945
Exact  Accuracy = 0.498
----------------
New Subjects Test
Valence Accuracy = 0.614
Arousal Accuracy = 0.600
Exact  Accuracy = 0.389
